# Создание и заполнение данных БД Postgre

In [1]:
%pip install python-dotenv psycopg2-binary pandas -q

In [2]:
import os
import json
import psycopg2
import pandas as pd
from psycopg2.extras import DictCursor
from dotenv import load_dotenv

# Получение секретов

In [3]:
# получаем текущую директорию ноутбука 
current_dir = os.getcwd()

# переходим на один уровень вверх
project_root = os.path.dirname(current_dir)

# формируем путь к файлу .env в папке Task1, там у нас лежит файл .env с настройками подключения к БД
dotenv_path = os.path.join(project_root, 'task_1_Docker', '.env')

# загружаем переменные окружения из указанного файла
load_dotenv(dotenv_path, override=True)

# получим доступ к переменным окружения
user = os.getenv("USER")
password = os.getenv("PASSWORD")
db_name = os.getenv("DB")
db_port = os.getenv("DB_PORT") 
secret_hash = os.getenv("SECRET_HASH") 

print(f"Загруженные данные: USER={user}, DB={db_name}, DB_PORT={db_port}")

Загруженные данные: USER=postgres, DB=user_logs_db, DB_PORT=5433


# Подключение к базе данных PostgreSQL

In [4]:
conn = None
try:
    conn = psycopg2.connect(
        host="localhost", # если Docker контейнер запущен локально, а ноутбук вне Docker.
                          # НО! если ноутбук также в Docker и в одной сети с БД,
                          # то нужно использовать имя сервиса Docker (например, 'db' или 'postgres_db').
        database=db_name,
        user=user,
        password=password,
        port=db_port
    )
    cursor = conn.cursor()

    print("Успешное подключение к базе данных!")
    
except Exception as e:
    print(f"Ошибка при подключении к базе данных: {e}")

Успешное подключение к базе данных!


In [5]:
# пример запроса
cursor.execute("SELECT version();")
db_version = cursor.fetchone()
print(f"Версия PostgreSQL: {db_version}")

Версия PostgreSQL: ('PostgreSQL 16.15 (Ubuntu 16.15-0ubuntu0.24.04.1) on x86_64-pc-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit',)


In [6]:
# получить список таблиц:
cursor.execute("""
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public'
""")
tables = cursor.fetchall()
print("\nТаблицы в базе данных:")
for table in tables:
    print(f"- {table[0]}")


Таблицы в базе данных:
- user_logs


In [7]:
# Соединение закроем в конце ноутбука, после выполнения заданий.

def query(sql, params=None):
    """Выполняет SELECT и возвращает результат в виде таблицы pandas."""
    with conn.cursor() as cur:
        cur.execute(sql, params)
        return pd.DataFrame(cur.fetchall(), columns=[c.name for c in cur.description])

# Если таблицы departments еще нет в БД - создадим ее из файла task_1_Docker/datasets/departments.csv
if query("SELECT to_regclass('public.departments') AS t").t[0] is None:
    with conn.cursor() as cur:
        cur.execute("CREATE TABLE departments (id INTEGER PRIMARY KEY, name VARCHAR(255))")
        with open(os.path.join(project_root, 'task_1_Docker', 'datasets', 'departments.csv'), encoding='utf-8') as f:
            cur.copy_expert("COPY departments FROM STDIN WITH (FORMAT csv, HEADER true)", f)
    conn.commit()
    print("Таблица departments создана")
print("Записей в user_logs:", query("SELECT COUNT(*) AS n FROM user_logs").n[0])
print("Кафедр в departments:", query("SELECT COUNT(*) AS n FROM departments").n[0])

Таблица departments создана
Записей в user_logs: 414528
Кафедр в departments: 43


Вам предоставлена БД с логами (действиями) студентов на образовательном портале за весенний семестр (агрегация по каждой неделе) по отдельному электронному курсу - таблица user_logs (примечание. создана в предыдущих л.р.).
- сourseid — уникальный идентификатор курса, дисциплины;
- userid — уникальный идентификатор студента (не используется в обучении);
- num_week — номер недели в году;
- s_all — количество всех событий на текущий момент;
- s_all_avg — среднее количество всех событий в неделю;
- s_course_viewed — количество просмотров курса;
- s_course_viewed_avg — среднее количество просмотров курса в неделю;
- s_q_attempt_viewed — количество просмотров теста;
- s_q_attempt_viewed_avg — среднее количество просмотров теста в неделю;
- s_a_course_module_viewed — количество просмотров модуля в курсе;
- s_a_course_module_viewed_avg — среднее количество просмотров модуля в курсе в неделю;
- s_a_submission_status_viewed — количество отправленных заданий на проверку;
- s_a_submission_status_viewed_avg — среднее количество ответов;
- namer_level — оценка за дисциплину;
- depart — номер кафедры;
- name_osno — основа обучения (имеет два значения: бюджет или контракт);
- name_formopril — форма обучения;
- leveled — уровень образования (имеет два значения: бакалавриат, магистратура);
- num_sem — номер семестра;
- kurs — номер курса учебной группы.

Также в таблице  departments хранятся названия кафедр, таблица связана с логами по полю depart:
id - код кафедры;
name - сокращенное название кафедры. 

## Задание 1 (если до этого еще этот шаг не был выполнен):

Измените данные вещественного типа, сейчас целая и дробная часть разделены запятой, замените ее на точку. 

Выведите первые 10 записей, чтобы проверить результат предобработки. 

In [8]:
# Колонки со средними значениями хранятся как текст с запятой ("4,5").
# Заменяем запятую на точку и меняем тип колонки на NUMERIC.
# Если колонка уже числовая (шаг выполнялся раньше) - пропускаем ее.
avg_columns = ['s_all_avg', 's_course_viewed_avg', 's_q_attempt_viewed_avg',
               's_a_course_module_viewed_avg', 's_a_submission_status_viewed_avg']

types = query("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'public' AND table_name = 'user_logs' AND column_name = ANY(%s)
""", (avg_columns,)).set_index('column_name').data_type

with conn.cursor() as cur:
    for col in avg_columns:
        if types[col] in ('character varying', 'text'):
            cur.execute(f"UPDATE user_logs SET {col} = REPLACE({col}, ',', '.') WHERE {col} LIKE '%%,%%'")
            print(f"{col}: заменено значений - {cur.rowcount}")
            cur.execute(f"ALTER TABLE user_logs ALTER COLUMN {col} TYPE NUMERIC USING {col}::NUMERIC")
        else:
            print(f"{col}: уже имеет тип {types[col]}, преобразование не требуется")
conn.commit()

s_all_avg: заменено значений - 302892


s_course_viewed_avg: заменено значений - 301977


s_q_attempt_viewed_avg: заменено значений - 63062


s_a_course_module_viewed_avg: заменено значений - 214722


s_a_submission_status_viewed_avg: заменено значений - 215341


In [9]:
# Проверка: первые 10 записей
query("SELECT * FROM user_logs LIMIT 10")

,courseid,userid,num_week,s_all,s_all_avg,s_course_viewed,s_course_viewed_avg,s_q_attempt_viewed,s_q_attempt_viewed_avg,s_a_course_module_viewed,...,s_a_submission_status_viewed_avg,namer_level,name_vatt,depart,name_osno,name_formopril,leveled,num_sem,kurs,date_vatt
0,71262,34527,6,9,9,4,4,0,0,0,...,0,3,Экзамен,22,1,1,1,2,2,18.06.2022
1,71262,34609,6,6,6,3,3,0,0,0,...,0,2,Экзамен,22,1,1,1,2,2,18.06.2022
2,71262,34610,6,13,13,5,5,0,0,1,...,1,5,Экзамен,22,1,1,1,2,2,18.06.2022
3,71262,34611,6,12,12,7,7,0,0,0,...,0,4,Экзамен,22,2,1,1,2,2,18.06.2022
4,71262,34612,6,24,24,8,8,0,0,0,...,0,3,Экзамен,22,1,1,1,2,2,18.06.2022
5,71262,34613,6,15,15,8,8,0,0,0,...,0,4,Экзамен,22,1,1,1,2,2,18.06.2022
6,71262,34615,6,14,14,7,7,0,0,0,...,0,4,Экзамен,22,1,1,1,2,2,18.06.2022
7,71262,34616,6,5,5,2,2,0,0,0,...,0,4,Экзамен,22,1,1,1,2,2,18.06.2022
8,71262,34616,8,2,3,2,2,0,0,0,...,0,4,Экзамен,22,1,1,1,2,2,18.06.2022
9,71262,34618,6,16,16,9,9,0,0,0,...,0,4,Экзамен,22,1,1,1,2,2,18.06.2022


In [10]:
# Типы колонок после преобразования
query("""
    SELECT column_name, data_type FROM information_schema.columns
    WHERE table_name = 'user_logs' AND column_name LIKE '%%avg'
""")

,column_name,data_type
0,s_all_avg,numeric
1,s_course_viewed_avg,numeric
2,s_q_attempt_viewed_avg,numeric
3,s_a_course_module_viewed_avg,numeric
4,s_a_submission_status_viewed_avg,numeric


**Результат:** в пяти колонках со средними значениями запятая заменена на точку, колонки переведены в тип NUMERIC.

## Задание 2: 

Выведите количество кафедр, за которыми закреплены курсы на портале.





In [11]:
query("SELECT COUNT(DISTINCT depart) AS departments_count FROM user_logs")

,departments_count
0,43


**Ответ:** курсы на портале закреплены за **43** кафедрами.

##  Задание 3:

Выведите сколько у каждой кафедры закреплено электронных курсов на портале. 
Требуется выводит сокращенное название кафедры и количество курсов. 
У какой кафедры больше всего курсов на портале?

In [12]:
dep_courses = query("""
    SELECT d.name                      AS department,
           COUNT(DISTINCT ul.courseid) AS courses_count
    FROM user_logs ul
    JOIN departments d ON d.id::TEXT = ul.depart::TEXT
    GROUP BY d.name
    ORDER BY courses_count DESC, department
""")
dep_courses

,department,courses_count
0,ДиСО,53
1,МиХТ,42
2,ПОиД,42
3,ГМиТТК,40
4,ЛиУТС,36
5,БИиИТ,35
6,ГМУиУП,35
7,АЭПиМ,33
8,ЛиП,33
9,ПиЭММО,33


In [13]:
top = dep_courses.iloc[0]
print(f"Больше всего курсов у кафедры {top.department}: {top.courses_count}")

Больше всего курсов у кафедры ДиСО: 53


**Ответ:** больше всего электронных курсов у кафедры **ДиСО – 53 курса**.

## Задание 4:

Ответьте на вопрос: существуют ли курсы, за которыми закреплено несколько кафедр? Если такие курсы есть, то выведите их количество.
Также выведите названия кафедр, которые совместно преподают один и тот же курс.




In [14]:
# Курсы, за которыми закреплено больше одной кафедры
query("""
    SELECT COUNT(*) AS courses_with_several_departments
    FROM (
        SELECT courseid
        FROM user_logs
        GROUP BY courseid
        HAVING COUNT(DISTINCT depart) > 1
    ) t
""")

,courses_with_several_departments
0,60


In [15]:
# Какие кафедры совместно ведут один и тот же курс
query("""
    SELECT ul.courseid,
           COUNT(DISTINCT d.name)                          AS departments_count,
           STRING_AGG(DISTINCT d.name, ', ' ORDER BY d.name) AS departments
    FROM user_logs ul
    JOIN departments d ON d.id::TEXT = ul.depart::TEXT
    GROUP BY ul.courseid
    HAVING COUNT(DISTINCT ul.depart) > 1
    ORDER BY departments_count DESC, ul.courseid
""")

,courseid,departments_count,departments
0,78057,25,"АСУ, АЭПиМ, ВИ, ВТиП, ГМДиОПИ, ГМУиУП, ГМиТТК,..."
1,72380,4,"ГМДиОПИ, ЛПиМ, МиХТ, ТОМ"
2,72402,4,"ГМДиОПИ, ЛПиМ, МиХТ, ТОМ"
3,72457,4,"ГМДиОПИ, ЛПиМ, МиХТ, ТОМ"
4,75833,4,"ГМДиОПИ, ГМиТТК, ЛиУТС, РМПИ"
5,79426,4,"ГМДиОПИ, ГМиТТК, ЛиУТС, РМПИ"
6,71495,3,"ГМиТТК, ПиСЗ, УиИС"
7,71571,3,"ЛиУТС, ПиСЗ, УиИС"
8,71892,3,"АЭПиМ, ТиЭС, ЭПП"
9,72359,3,"ЛПиМ, МиХТ, ТОМ"


**Ответ:** да, такие курсы есть – **60 курсов** закреплены за несколькими кафедрами. Больше всего кафедр (25) у курса 78057; список кафедр по каждому курсу выведен в таблице выше.

## Задание 5:

Выведите количество студентов, которые получили 2, 3, 4, 5.

Пример вывода:

| namer_level |	count |
|-----|------|
|2 |	4 |
|3 |	3435 |
|4 | 	4676765|
|5 | 232 |


In [16]:
# Студент учитывается один раз для каждой оценки, которую он получил (по любому курсу)
query("""
    SELECT namer_level, COUNT(DISTINCT userid) AS count
    FROM user_logs
    GROUP BY namer_level
    ORDER BY namer_level
""")

,namer_level,count
0,2,1069
1,3,1884
2,4,3243
3,5,3407


**Ответ:** оценку 2 получили 1069 студентов, 3 – 1884, 4 – 3243, 5 – 3407 (студент учитывается в каждой оценке, которую получал по разным курсам).

## Задание 6:

Выведите студента, который больше всех работает на портале (у него максимальное количество логов за вест период обучения).

In [17]:
# s_all - число событий студента за неделю; суммируем за весь период по всем курсам
query("""
    SELECT userid, SUM(s_all) AS total_events
    FROM user_logs
    GROUP BY userid
    ORDER BY total_events DESC
    LIMIT 1
""")

,userid,total_events
0,28871,10300


**Ответ:** больше всех работает на портале студент **28871** – 10 300 событий за весь период.

## Задание 7:

Выведите по каждой недели среднее количество всех событий на портале.

In [18]:
query("""
    SELECT num_week, ROUND(AVG(s_all), 2) AS avg_s_all
    FROM user_logs
    GROUP BY num_week
    ORDER BY num_week
""")

,num_week,avg_s_all
0,6,13.80
1,7,9.62
2,8,8.03
3,9,9.39
4,10,8.21
5,11,10.02
6,12,9.38
7,13,10.01
8,14,9.86
9,15,10.35


**Вывод:** активность растет к концу семестра, максимум приходится на 22–24 недели (подготовка к сессии), после 25 недели резко падает.

## Задание 8: 

Выведите название кафедры, у которой больше всего отличников.

Отдельно выведите название кафедры, у которой больше всего двоечников. 

In [19]:
# Отличник - студент с оценкой 5 по курсу кафедры; считаем уникальных студентов
query("""
    SELECT d.name AS department, COUNT(DISTINCT ul.userid) AS excellent_students
    FROM user_logs ul
    JOIN departments d ON d.id::TEXT = ul.depart::TEXT
    WHERE ul.namer_level::TEXT = '5'
    GROUP BY d.name
    ORDER BY excellent_students DESC
    LIMIT 1
""")

,department,excellent_students
0,ДиСО,310


In [20]:
query("""
    SELECT d.name AS department, COUNT(DISTINCT ul.userid) AS failing_students
    FROM user_logs ul
    JOIN departments d ON d.id::TEXT = ul.depart::TEXT
    WHERE ul.namer_level::TEXT = '2'
    GROUP BY d.name
    ORDER BY failing_students DESC
    LIMIT 1
""")

,department,failing_students
0,Эконом.,72


**Ответ:** больше всего отличников у кафедры **ДиСО (310 студентов)**, больше всего двоечников – у кафедры **Эконом. (72 студента)**.

## Задание 9:
Провести анализ пиковой активности студентов перед экзаменом (с использованием (Common Table Expression — CTE), оператор with).

Вывести, на какой неделе семестра студенты проявляли наибольшую активность в курсе в целом, и как эта активность распределяется между студентами-бюджетниками и контрактниками.

Пример вывода :

| name_osno | week_number	| avg_s_all	| avg_s_course_viewed |	avg_s_q_attempt_viewed |
|-----|------|------|------|------|
| бюджет |	14	| 125.45 |	45.67 |	32.12 |
|контракт |	14	| 98.76 |	38.90 |	25.43 |

In [21]:
# name_osno: 1 - бюджет, 2 - контракт
query("""
    WITH weekly AS (                    -- средняя активность по каждой неделе
        SELECT num_week, AVG(s_all) AS avg_s_all
        FROM user_logs
        GROUP BY num_week
    ),
    peak_week AS (                      -- неделя с максимальной активностью
        SELECT num_week
        FROM weekly
        ORDER BY avg_s_all DESC
        LIMIT 1
    )
    SELECT CASE ul.name_osno::TEXT WHEN '1' THEN 'бюджет' WHEN '2' THEN 'контракт'
                ELSE ul.name_osno::TEXT END           AS name_osno,
           ul.num_week                                AS week_number,
           ROUND(AVG(ul.s_all), 2)                    AS avg_s_all,
           ROUND(AVG(ul.s_course_viewed), 2)          AS avg_s_course_viewed,
           ROUND(AVG(ul.s_q_attempt_viewed), 2)       AS avg_s_q_attempt_viewed
    FROM user_logs ul
    JOIN peak_week p ON p.num_week = ul.num_week
    GROUP BY 1, 2
    ORDER BY 1
""")

,name_osno,week_number,avg_s_all,avg_s_course_viewed,avg_s_q_attempt_viewed
0,бюджет,24,20.79,3.78,4.94
1,контракт,24,28.69,4.73,5.42


**Ответ:** пик активности – **24 неделя** (перед экзаменами). На этой неделе контрактники в среднем активнее бюджетников: 28,69 событий против 20,79, у них также больше просмотров курса и тестов.

In [22]:
# закрытие соединения с БД
cursor.close()
conn.close()